In [3]:
!pip install python-dotenv

In [ ]:
import os
import json
import time
from datetime import datetime
import dotenv
from huggingface_hub import InferenceClient  # Updated from InferenceApi
import re
import logging
from typing import List, Dict, Any, Optional

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("chatbot.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("mental_health_chatbot")

# Load environment variables from .env file
dotenv.load_dotenv()

# Get API token from environment variable
API_TOKEN = os.getenv("HUGGINGFACE_API_TOKEN")
if not API_TOKEN:
    raise EnvironmentError("Please set the HUGGINGFACE_API_TOKEN environment variable")

# Model configuration
MODEL_ID = "meta-llama/Llama-2-13b-chat-hf"
MAX_RETRIES = 3
RETRY_DELAY = 2  # seconds

# Initialize Hugging Face Inference Client (updated from InferenceApi)
client = InferenceClient(model=MODEL_ID, token=API_TOKEN)

# Crisis detection patterns
CRISIS_PATTERNS = [
    r"(?i)(suicid(e|al))",
    r"(?i)(kill\s+(my)?self)",
    r"(?i)(end\s+(my)?\s+life)",
    r"(?i)(harm(ing)?\s+(my)?self)",
    r"(?i)(want\s+to\s+die)",
    r"(?i)(don't\s+want\s+to\s+(live|be\s+alive))"
]

# Crisis resources
CRISIS_RESOURCES = """
If you're experiencing thoughts of harming yourself or feeling suicidal, please reach out for help immediately:

- National Suicide Prevention Lifeline: 988 or 1-800-273-8255
- Crisis Text Line: Text HOME to 741741
- International Association for Suicide Prevention: https://www.iasp.info/resources/Crisis_Centres/

Remember, you're not alone, and professional help is available.
"""

class ConversationHistory:
    """Manage conversation history for context retention."""

    def __init__(self, max_history: int = 5):
        self.history: List[Dict[str, str]] = []
        self.max_history = max_history

    def add_exchange(self, user_message: str, bot_response: str) -> None:
        """Add a message exchange to history."""
        timestamp = datetime.now().isoformat()
        self.history.append({
            "timestamp": timestamp,
            "user": user_message,
            "bot": bot_response
        })

        # Keep history within max length
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]

    def get_context_for_prompt(self) -> str:
        """Format history for inclusion in the prompt."""
        if not self.history:
            return ""

        context = "Previous conversation:\n"
        for exchange in self.history:
            context += f"User: {exchange['user']}\n"
            context += f"Assistant: {exchange['bot']}\n"
        return context

    def save_to_file(self, user_id: str = "anonymous") -> None:
        """Save conversation history to a file."""
        if not self.history:
            return

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"conversation_{user_id}_{timestamp}.json"

        try:
            with open(filename, "w") as f:
                json.dump(self.history, f, indent=2)
            logger.info(f"Conversation saved to {filename}")
        except Exception as e:
            logger.error(f"Failed to save conversation: {e}")

def detect_crisis(text: str) -> bool:
    """
    Detect potential crisis indicators in user input.
    """
    for pattern in CRISIS_PATTERNS:
        if re.search(pattern, text):
            return True
    return False

def get_chatbot_response(user_input: str, conversation_history: ConversationHistory) -> str:
    """
    Generate a chatbot response for mental health support with context from conversation history.
    """
    # Check for crisis indicators
    if detect_crisis(user_input):
        logger.warning("Crisis indicators detected in user input")
        return f"I'm concerned about what you've shared. {CRISIS_RESOURCES}"

    # Build prompt with conversation history context
    context = conversation_history.get_context_for_prompt()

    prompt = (
        "<s>[INST] <<SYS>> You are a compassionate mental health support chatbot. "
        "Offer supportive and empathetic responses to help users feel better. "
        "Acknowledge emotions, provide encouragement, and suggest healthy coping strategies. "
        "Remember that you are not a replacement for professional mental health care. "
        "If someone seems to be in crisis, encourage them to seek immediate professional help. "
        f"{context}<</SYS>> {user_input} [/INST]"
    )

    # Try to get a response with retries
    for attempt in range(MAX_RETRIES):
        try:
            logger.debug(f"Sending request to model (attempt {attempt+1}/{MAX_RETRIES})")

            # Using the new client's text_generation method
            response_text = client.text_generation(
                prompt,
                max_new_tokens=512,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1
            )

            # Process the response to extract just the assistant's reply
            if response_text:
                # Extract the part after [/INST]
                parts = response_text.split('[/INST]')
                if len(parts) > 1:
                    response_text = parts[-1].strip()

                # Add disclaimer if not already present in response
                disclaimer = "\n\n(I'm an AI assistant, not a licensed therapist. For professional help, please consult a mental health professional.)"
                if "not a licensed therapist" not in response_text.lower():
                    response_text += disclaimer

                return response_text
            else:
                logger.warning(f"Empty response received from API")
                if attempt < MAX_RETRIES - 1:
                    time.sleep(RETRY_DELAY)

        except Exception as e:
            logger.error(f"Error during API call: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY)

    return "I'm having trouble responding right now. If you're feeling distressed, please consider reaching out to a mental health professional or a trusted person in your life."

def chatbot_loop() -> None:
    """
    Start an interactive chatbot loop with conversation history.
    """
    conversation = ConversationHistory()

    welcome_message = (
        "Mental Health Chatbot: Hello! I'm here to provide support and a listening ear. "
        "How are you feeling today?\n\n"
        "Type 'exit' or 'quit' to end our conversation. Remember, I'm not a replacement "
        "for professional help."
    )
    print(welcome_message)

    try:
        while True:
            user_message = input("\nYou: ").strip()

            if user_message.lower() in {"exit", "quit", "bye", "goodbye"}:
                farewell = "Mental Health Chatbot: Take care of yourself. Remember that it's okay to ask for help when you need it. I'm here whenever you want to talk again."
                print(farewell)
                conversation.add_exchange(user_message, farewell)
                conversation.save_to_file()
                break

            # Get response
            print("Mental Health Chatbot: Thinking...")
            response = get_chatbot_response(user_message, conversation)
            print(f"Mental Health Chatbot: {response}")

            # Update conversation history
            conversation.add_exchange(user_message, response)

    except KeyboardInterrupt:
        print("\nConversation ended. Taking care of your mental health is important!")
        conversation.save_to_file()
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        print("\nI apologize, but I've encountered an error. Please try again later.")
        conversation.save_to_file()

if __name__ == "__main__":
    print(f"Starting Mental Health Support Chatbot using {MODEL_ID}")
    chatbot_loop()